**biến file GeoJSON thành file chỉ có nước**

In [9]:
import json

def extract_water_regions(input_geojson_path, output_geojson_path):
    """
    Extracts features identified as water regions from a GeoJSON file and saves them to a new file.
    Assumes water regions have properties like 'natural' == 'water', 'water' == 'lake', 'waterway' etc.
    """
    with open(input_geojson_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    features = data.get('features', [])
    water_features = []
    
    for feature in features:
        props = feature.get('properties', {})
        # Check standard OSM tags for water and waterways
        if (props.get('natural') == 'water' or
            props.get('water') is not None or
            props.get('waterway') is not None or
            props.get('landuse') in ['reservoir', 'basin']):
            water_features.append(feature)
            
    filtered_data = {
        "type": "FeatureCollection",
        "features": water_features
    }
    
    with open(output_geojson_path, 'w', encoding='utf-8') as f:
        json.dump(filtered_data, f, ensure_ascii=False, indent=2)
        
    print(f"Extracted {len(water_features)} water regions from {input_geojson_path}")
    print(f"Saved filtered data to {output_geojson_path}")

# Example usage:
input_file = "planet_105.755,21.114_105.825,21.158.osm.geojson"
output_file = "water_only.geojson"
extract_water_regions(input_file, output_file)


Extracted 82 water regions from planet_105.755,21.114_105.825,21.158.osm.geojson
Saved filtered data to water_only.geojson


**sinh file graph nối các điểm và loại bỏ các edge không hợp lệ**

In [12]:
import json
import math
import time
from itertools import combinations
from pathlib import Path

import graph_tool.all as gt
from shapely.geometry import LineString, mapping, shape

try:
    from shapely.strtree import STRtree
except Exception:
    STRtree = None

from IPython.display import IFrame, display

ROOT = Path(".")
INPUT_GEOJSON = ROOT / "hodongda.geojson"
GRAPH_GT_PATH = ROOT / "graph.gt"
GRAPH_JSON_PATH = ROOT / "data.json"
NODES_GEOJSON_PATH = ROOT / "nodes.geojson"
VALID_EDGES_GEOJSON_PATH = ROOT / "edges_valid.geojson"
BLOCKED_EDGES_GEOJSON_PATH = ROOT / "edges_blocked.geojson"
LAND_GEOJSON_PATH = ROOT / "land.geojson"
WATER_GEOJSON_PATH = ROOT / "water.geojson"
MAP_HTML_PATH = ROOT / "leaflet.html"


def iter_polygons(geom):
    if geom.is_empty:
        return

    if geom.geom_type == "Polygon":
        yield geom
    elif geom.geom_type == "MultiPolygon":
        for poly in geom.geoms:
            yield poly
    elif geom.geom_type == "GeometryCollection":
        for sub_geom in geom.geoms:
            yield from iter_polygons(sub_geom)


def normalize_ring(coords):
    ring = [(float(coord[0]), float(coord[1])) for coord in coords]

    if len(ring) >= 2 and ring[0] == ring[-1]:
        ring = ring[:-1]

    cleaned = []
    for point in ring:
        if not cleaned or point != cleaned[-1]:
            cleaned.append(point)

    if len(set(cleaned)) < 3:
        return []

    return cleaned


def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


start = time.time()
source = json.loads(INPUT_GEOJSON.read_text(encoding="utf-8"))

rings = []
land_polygons = []
water_polygons = []

for feature in source.get("features", []):
    geometry = feature.get("geometry")
    if not geometry:
        continue

    geom = shape(geometry)
    properties = feature.get("properties") or {}
    is_water = str(properties.get("natural", "")).lower() == "water"

    if not geom.is_valid:
        geom = geom.buffer(0)

    for polygon in iter_polygons(geom):
        if polygon.is_empty:
            continue

        if is_water:
            water_polygons.append(polygon)
        else:
            land_polygons.append(polygon)

        ext_ring = normalize_ring(polygon.exterior.coords)
        if ext_ring:
            rings.append(ext_ring)

        for interior in polygon.interiors:
            int_ring = normalize_ring(interior.coords)
            if int_ring:
                rings.append(int_ring)

coord_to_id = {}
nodes = []
for ring in rings:
    for coord in ring:
        if coord not in coord_to_id:
            coord_to_id[coord] = len(nodes)
            nodes.append(coord)

num_nodes = len(nodes)
num_candidate_edges = num_nodes * (num_nodes - 1) // 2

land_tree = STRtree(land_polygons) if (STRtree and land_polygons) else None


def resolve_land_candidates(query_result):
    candidates = []
    for item in query_result:
        if hasattr(item, "geom_type"):
            candidates.append(item)
        else:
            candidates.append(land_polygons[int(item)])
    return candidates


def blocks_edge(line):
    if not land_polygons:
        return False

    if land_tree is not None:
        candidate_polys = resolve_land_candidates(land_tree.query(line))
    else:
        candidate_polys = land_polygons

    for poly in candidate_polys:
        if not line.intersects(poly):
            continue

        # Boundary-only contact is allowed by requirement.
        if not line.touches(poly):
            return True

    return False


valid_edges = []
blocked_edges = []
valid_lengths = []
blocked_lengths = []

for src, dst in combinations(range(num_nodes), 2):
    p1 = nodes[src]
    p2 = nodes[dst]
    line = LineString([p1, p2])
    length_deg = math.hypot(p1[0] - p2[0], p1[1] - p2[1])

    if blocks_edge(line):
        blocked_edges.append((src, dst))
        blocked_lengths.append(length_deg)
    else:
        valid_edges.append((src, dst))
        valid_lengths.append(length_deg)

# Persist graph-tool graph (valid edges only).
graph = gt.Graph(directed=False)
graph.add_vertex(num_nodes)

vp_lon = graph.new_vertex_property("double")
vp_lat = graph.new_vertex_property("double")
for idx, (lon, lat) in enumerate(nodes):
    vertex = graph.vertex(idx)
    vp_lon[vertex] = lon
    vp_lat[vertex] = lat

graph.vp["lon"] = vp_lon
graph.vp["lat"] = vp_lat

ep_length = graph.new_edge_property("double")
for (src, dst), length_deg in zip(valid_edges, valid_lengths):
    edge = graph.add_edge(src, dst)
    ep_length[edge] = length_deg

graph.ep["length_deg"] = ep_length
graph.save(str(GRAPH_GT_PATH))

nodes_payload = [{"id": idx, "lon": lon, "lat": lat} for idx, (lon, lat) in enumerate(nodes)]

edges_payload = [
    {"source": src, "target": dst, "length_deg": length_deg}
    for (src, dst), length_deg in zip(valid_edges, valid_lengths)
]

graph_data = {
    "nodes": nodes_payload,
    "edges": edges_payload,
    "meta": {
        "input_file": INPUT_GEOJSON.name,
        "num_vertices": num_nodes,
        "num_candidate_edges": num_candidate_edges,
        "num_edges": len(valid_edges),
        "num_blocked_edges": len(blocked_edges),
        "rendered_edges": len(valid_edges),
        "sampled_edges": False,
        "max_edges_visible": 5000,
        "land_rule": "polygon is land blocker unless natural==water",
        "edge_rule": "remove when intersects land interior; boundary touch allowed",
    },
}
write_json(GRAPH_JSON_PATH, graph_data)


def build_node_geojson():
    return {
        "type": "FeatureCollection",
        "features": [
            {
                "type": "Feature",
                "geometry": {"type": "Point", "coordinates": [lon, lat]},
                "properties": {"id": idx},
            }
            for idx, (lon, lat) in enumerate(nodes)
        ],
    }


def build_edge_geojson(edge_list, length_list, blocked):
    features = []
    for (src, dst), length_deg in zip(edge_list, length_list):
        lon1, lat1 = nodes[src]
        lon2, lat2 = nodes[dst]
        features.append(
            {
                "type": "Feature",
                "geometry": {
                    "type": "LineString",
                    "coordinates": [[lon1, lat1], [lon2, lat2]],
                },
                "properties": {
                    "source": src,
                    "target": dst,
                    "length_deg": length_deg,
                    "blocked": blocked,
                    "reason": "land_interior_intersection" if blocked else "valid",
                },
            }
        )

    return {"type": "FeatureCollection", "features": features}


land_geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": mapping(poly),
            "properties": {"kind": "land_blocker"},
        }
        for poly in land_polygons
    ],
}

water_geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": mapping(poly),
            "properties": {"kind": "water"},
        }
        for poly in water_polygons
    ],
}

nodes_geojson = build_node_geojson()
valid_edges_geojson = build_edge_geojson(valid_edges, valid_lengths, blocked=False)
blocked_edges_geojson = build_edge_geojson(blocked_edges, blocked_lengths, blocked=True)

write_json(NODES_GEOJSON_PATH, nodes_geojson)
write_json(VALID_EDGES_GEOJSON_PATH, valid_edges_geojson)
write_json(BLOCKED_EDGES_GEOJSON_PATH, blocked_edges_geojson)
write_json(LAND_GEOJSON_PATH, land_geojson)
write_json(WATER_GEOJSON_PATH, water_geojson)

if nodes:
    center_lat = sum(lat for _, lat in nodes) / len(nodes)
    center_lon = sum(lon for lon, _ in nodes) / len(nodes)
else:
    center_lat = 0.0
    center_lon = 0.0

html_template = """<!doctype html>
<html lang=\"en\">
<head>
  <meta charset=\"utf-8\" />
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\" />
  <title>Polygon Vertex Graph</title>
  <link rel=\"stylesheet\" href=\"https://unpkg.com/leaflet@1.9.4/dist/leaflet.css\" integrity=\"sha256-p4NxAoJBhIIN+hmNHrzRCf9tD/miZyoHS5obTRR9BMY=\" crossorigin=\"\"/>
  <style>
    html, body, #map { height: 100%; margin: 0; }
    .stats {
      background: rgba(255, 255, 255, 0.95);
      padding: 8px 10px;
      border-radius: 6px;
      box-shadow: 0 1px 6px rgba(0,0,0,0.2);
      line-height: 1.35;
      font: 12px/1.35 system-ui, sans-serif;
    }
  </style>
</head>
<body>
  <div id=\"map\"></div>
  <script src=\"https://unpkg.com/leaflet@1.9.4/dist/leaflet.js\" integrity=\"sha256-20nQCchB9co0qIjJZRGuk2/Z9VM+kNiyxNV1lvTlZBo=\" crossorigin=\"\"></script>
  <script>
    const nodesGeoJson = __NODES_GEOJSON__;
    const validEdgesGeoJson = __VALID_EDGES_GEOJSON__;
    const blockedEdgesGeoJson = __BLOCKED_EDGES_GEOJSON__;
    const landGeoJson = __LAND_GEOJSON__;
    const waterGeoJson = __WATER_GEOJSON__;
    const meta = __META_JSON__;

    const map = L.map('map', { preferCanvas: true }).setView([__CENTER_LAT__, __CENTER_LON__], 17);

    L.tileLayer('https://tile.openstreetmap.org/{z}/{x}/{y}.png', {
      maxZoom: 22,
      attribution: '&copy; OpenStreetMap contributors'
    }).addTo(map);

    const waterLayer = L.geoJSON(waterGeoJson, {
      style: { color: '#2a7ab9', weight: 1, fillColor: '#64b5f6', fillOpacity: 0.25 }
    }).addTo(map);

    const landLayer = L.geoJSON(landGeoJson, {
      style: { color: '#8d6e63', weight: 1, fillColor: '#c8ad7f', fillOpacity: 0.45 }
    }).addTo(map);

    const validEdgesLayer = L.geoJSON(validEdgesGeoJson, {
      style: { color: '#0b9a6f', weight: 1, opacity: 0.8 }
    }).addTo(map);

    const blockedEdgesLayer = L.geoJSON(blockedEdgesGeoJson, {
      style: { color: '#d73a49', weight: 1.2, opacity: 0.95, dashArray: '4,4' }
    }).addTo(map);

    const nodesLayer = L.geoJSON(nodesGeoJson, {
      pointToLayer: function(feature, latlng) {
        return L.circleMarker(latlng, {
          radius: 4,
          color: '#111',
          weight: 1,
          fillColor: '#ffd54f',
          fillOpacity: 0.95
        });
      },
      onEachFeature: function(feature, layer) {
        layer.bindTooltip('node ' + feature.properties.id);
      }
    }).addTo(map);

    const layers = {
      'Water polygons': waterLayer,
      'Land blockers': landLayer,
      'Valid edges': validEdgesLayer,
      'Blocked edges': blockedEdgesLayer,
      'Vertices': nodesLayer
    };
    L.control.layers(null, layers, { collapsed: false }).addTo(map);

    const stats = L.control({ position: 'topright' });
    stats.onAdd = function() {
      const div = L.DomUtil.create('div', 'stats');
      div.innerHTML = [
        '<strong>Graph Summary</strong>',
        'vertices: ' + meta.num_vertices,
        'candidate edges: ' + meta.num_candidate_edges,
        'valid edges: ' + meta.num_edges,
        'blocked edges: ' + meta.num_blocked_edges,
        'land rule: not natural==water',
        'edge rule: keep boundary touch'
      ].join('<br/>');
      return div;
    };
    stats.addTo(map);

    const fitGroup = L.featureGroup([waterLayer, landLayer, validEdgesLayer, blockedEdgesLayer, nodesLayer]);
    if (fitGroup.getBounds().isValid()) {
      map.fitBounds(fitGroup.getBounds().pad(0.08));
    }
  </script>
</body>
</html>
"""

leaflet_html = (
    html_template
    .replace("__NODES_GEOJSON__", json.dumps(nodes_geojson))
    .replace("__VALID_EDGES_GEOJSON__", json.dumps(valid_edges_geojson))
    .replace("__BLOCKED_EDGES_GEOJSON__", json.dumps(blocked_edges_geojson))
    .replace("__LAND_GEOJSON__", json.dumps(land_geojson))
    .replace("__WATER_GEOJSON__", json.dumps(water_geojson))
    .replace("__META_JSON__", json.dumps(graph_data["meta"]))
    .replace("__CENTER_LAT__", f"{center_lat:.10f}")
    .replace("__CENTER_LON__", f"{center_lon:.10f}")
)

MAP_HTML_PATH.write_text(leaflet_html, encoding="utf-8")

elapsed = time.time() - start
print(f"Input file: {INPUT_GEOJSON.name}")
print(f"Vertices: {num_nodes}")
print(f"Candidate edges: {num_candidate_edges}")
print(f"Valid edges: {len(valid_edges)}")
print(f"Blocked edges: {len(blocked_edges)}")
print(f"Saved graph: {GRAPH_GT_PATH}")
print(f"Saved graph data: {GRAPH_JSON_PATH}")
print(f"Saved map: {MAP_HTML_PATH}")
print(f"Elapsed seconds: {elapsed:.3f}")

display(IFrame(src=str(MAP_HTML_PATH), width="100%", height=620))

Input file: hodongda.geojson
Vertices: 78
Candidate edges: 3003
Valid edges: 1812
Blocked edges: 1191
Saved graph: graph.gt
Saved graph data: data.json
Saved map: leaflet.html
Elapsed seconds: 0.602


**refactor file json**

In [13]:
# Validation: reload persisted graph and confirm counts against exported JSON.
import json
import graph_tool.all as gt

loaded_graph = gt.load_graph("graph.gt")
with open("data.json", "r", encoding="utf-8") as f:
    exported = json.load(f)

print("Loaded graph vertices:", loaded_graph.num_vertices())
print("Loaded graph edges:", loaded_graph.num_edges())
print("Exported vertices:", exported["meta"]["num_vertices"])
print("Exported valid edges:", exported["meta"]["num_edges"])

assert loaded_graph.num_vertices() == exported["meta"]["num_vertices"]
assert loaded_graph.num_edges() == exported["meta"]["num_edges"]
print("Validation passed.")

Loaded graph vertices: 78
Loaded graph edges: 1812
Exported vertices: 78
Exported valid edges: 1812
Validation passed.


In [14]:
import heapq
import json
import math
import time
from pathlib import Path

import graph_tool.all as gt
from IPython.display import IFrame, display

ASTAR_DEBUG_HTML_PATH = Path("astar_debug_map.html")
ASTAR_DEBUG_JSON_PATH = Path("astar_debug_data.json")


def ensure_graph_and_nodes():
    g = globals().get("graph")
    if g is None or int(g.num_vertices()) == 0:
        g = gt.load_graph(str(GRAPH_GT_PATH))

    lon_prop = g.vp["lon"]
    lat_prop = g.vp["lat"]
    node_coords = [(float(lon_prop[v]), float(lat_prop[v])) for v in g.vertices()]

    length_prop = g.ep["length_deg"] if "length_deg" in g.ep else None
    return g, node_coords, length_prop


def euclidean_heuristic(node_id, goal_id, coords):
    x, y = coords[node_id]
    gx, gy = coords[goal_id]
    return math.hypot(gx - x, gy - y)


def manhattan_heuristic(node_id, goal_id, coords):
    x, y = coords[node_id]
    gx, gy = coords[goal_id]
    return abs(gx - x) + abs(gy - y)


def build_heuristic(coords, heuristic=None, heuristic_expr=None):
    # Priority: explicit callable > expression string > named default.
    if callable(heuristic):
        return heuristic, "callable"

    expr = heuristic_expr if heuristic_expr is not None else heuristic

    if expr is None:
        return lambda n, g: euclidean_heuristic(n, g, coords), "euclidean"

    if isinstance(expr, str):
        expr = expr.strip()
        lowered = expr.lower()
        if lowered in {"", "euclidean", "l2"}:
            return lambda n, g: euclidean_heuristic(n, g, coords), "euclidean"
        if lowered in {"zero", "dijkstra"}:
            return lambda n, g: 0.0, "zero"
        if lowered in {"manhattan", "l1"}:
            return lambda n, g: manhattan_heuristic(n, g, coords), "manhattan"

        allowed_math = {
            "abs": abs,
            "sqrt": math.sqrt,
            "pow": pow,
            "sin": math.sin,
            "cos": math.cos,
            "tan": math.tan,
            "asin": math.asin,
            "acos": math.acos,
            "atan": math.atan,
            "atan2": math.atan2,
            "exp": math.exp,
            "log": math.log,
            "log10": math.log10,
            "pi": math.pi,
            "e": math.e,
            "min": min,
            "max": max,
        }

        code = compile(expr, "<heuristic_expr>", "eval")
        blocked_names = set(code.co_names) - {
            "x",
            "y",
            "gx",
            "gy",
            "dx",
            "dy",
            "node",
            "goal",
            *allowed_math.keys(),
        }
        if blocked_names:
            raise ValueError(
                "Unsupported names in heuristic expression: "
                + ", ".join(sorted(blocked_names))
            )

        def expr_heuristic(node_id, goal_id):
            x, y = coords[node_id]
            gx, gy = coords[goal_id]
            local_env = {
                "x": x,
                "y": y,
                "gx": gx,
                "gy": gy,
                "dx": abs(gx - x),
                "dy": abs(gy - y),
                "node": node_id,
                "goal": goal_id,
            }
            local_env.update(allowed_math)
            value = eval(code, {"__builtins__": {}}, local_env)
            return float(value)

        return expr_heuristic, f"expr:{expr}"

    raise TypeError("heuristic must be callable or string expression")


def edge_cost(u, v, coords, length_prop=None, edge_obj=None):
    if length_prop is not None and edge_obj is not None:
        return float(length_prop[edge_obj])
    x1, y1 = coords[u]
    x2, y2 = coords[v]
    return math.hypot(x2 - x1, y2 - y1)


def pick_farthest_nodes(coords):
    best = (0, 0)
    best_d = -1.0
    for i in range(len(coords)):
        x1, y1 = coords[i]
        for j in range(i + 1, len(coords)):
            x2, y2 = coords[j]
            d = (x2 - x1) ** 2 + (y2 - y1) ** 2
            if d > best_d:
                best_d = d
                best = (i, j)
    return best


def astar_search(
    g,
    coords,
    start_id,
    goal_id,
    heuristic=None,
    heuristic_expr=None,
    length_prop=None,
    max_expansions=None,
):
    h_fn, h_name = build_heuristic(coords, heuristic=heuristic, heuristic_expr=heuristic_expr)

    start_id = int(start_id)
    goal_id = int(goal_id)

    open_heap = []
    push_count = 0
    g_score = {start_id: 0.0}
    h_score = {start_id: float(h_fn(start_id, goal_id))}
    f_score = {start_id: g_score[start_id] + h_score[start_id]}
    parent = {start_id: None}

    open_set = {start_id}
    closed_set = set()
    closed_order = []
    relaxations = []

    heapq.heappush(open_heap, (f_score[start_id], push_count, start_id))

    expansions = 0
    t0 = time.time()

    while open_heap:
        _, _, current = heapq.heappop(open_heap)
        if current in closed_set:
            continue

        open_set.discard(current)
        closed_set.add(current)
        closed_order.append(current)

        if current == goal_id:
            break

        expansions += 1
        if max_expansions is not None and expansions > int(max_expansions):
            break

        vertex = g.vertex(current)
        for edge in vertex.all_edges():
            u = int(edge.source())
            v = int(edge.target())
            neighbor = v if u == current else u

            if neighbor in closed_set:
                continue

            step_cost = edge_cost(current, neighbor, coords, length_prop=length_prop, edge_obj=edge)
            tentative_g = g_score[current] + step_cost

            if tentative_g >= g_score.get(neighbor, float("inf")):
                continue

            parent[neighbor] = current
            g_score[neighbor] = tentative_g
            h_val = float(h_fn(neighbor, goal_id))
            h_score[neighbor] = h_val
            f_val = tentative_g + h_val
            f_score[neighbor] = f_val

            push_count += 1
            heapq.heappush(open_heap, (f_val, push_count, neighbor))
            open_set.add(neighbor)

            relaxations.append(
                {
                    "from": current,
                    "to": neighbor,
                    "g": tentative_g,
                    "h": h_val,
                    "f": f_val,
                    "step_cost": step_cost,
                }
            )

    elapsed = time.time() - t0

    path = []
    if goal_id in parent:
        cur = goal_id
        while cur is not None:
            path.append(cur)
            cur = parent.get(cur)
        path.reverse()

    path_edges = list(zip(path[:-1], path[1:])) if len(path) > 1 else []
    path_length = g_score.get(goal_id)

    score_table = {
        int(node): {
            "g": float(g_score.get(node, float("inf"))),
            "h": float(h_score.get(node, float("inf"))),
            "f": float(f_score.get(node, float("inf"))),
            "parent": None if parent.get(node) is None else int(parent[node]),
            "closed": node in closed_set,
            "open": node in open_set,
        }
        for node in set(g_score.keys()) | set(open_set) | set(closed_set)
    }

    return {
        "heuristic_name": h_name,
        "start": start_id,
        "goal": goal_id,
        "elapsed_sec": elapsed,
        "expanded_nodes": len(closed_order),
        "closed_order": closed_order,
        "open_remaining": sorted(open_set),
        "relaxations": relaxations,
        "path": path,
        "path_edges": path_edges,
        "path_length": None if path_length is None else float(path_length),
        "scores": score_table,
    }


def node_feature(node_id, coords, props):
    lon, lat = coords[node_id]
    return {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [lon, lat]},
        "properties": props,
    }


def edge_feature(src, dst, coords, props):
    lon1, lat1 = coords[src]
    lon2, lat2 = coords[dst]
    return {
        "type": "Feature",
        "geometry": {
            "type": "LineString",
            "coordinates": [[lon1, lat1], [lon2, lat2]],
        },
        "properties": props,
    }


def build_debug_geojson(result, coords):
    route_features = []
    for idx, (src, dst) in enumerate(result["path_edges"]):
        route_features.append(
            edge_feature(
                src,
                dst,
                coords,
                {
                    "step": idx,
                    "source": src,
                    "target": dst,
                },
            )
        )

    relaxation_features = []
    for idx, item in enumerate(result["relaxations"]):
        relaxation_features.append(
            edge_feature(
                item["from"],
                item["to"],
                coords,
                {
                    "idx": idx,
                    "from": item["from"],
                    "to": item["to"],
                    "g": item["g"],
                    "h": item["h"],
                    "f": item["f"],
                    "step_cost": item["step_cost"],
                },
            )
        )

    closed_features = []
    for order, node_id in enumerate(result["closed_order"]):
        score = result["scores"].get(node_id, {})
        closed_features.append(
            node_feature(
                node_id,
                coords,
                {
                    "id": node_id,
                    "order": order,
                    "g": score.get("g"),
                    "h": score.get("h"),
                    "f": score.get("f"),
                    "state": "closed",
                },
            )
        )

    open_features = []
    for node_id in result["open_remaining"]:
        score = result["scores"].get(node_id, {})
        open_features.append(
            node_feature(
                node_id,
                coords,
                {
                    "id": node_id,
                    "g": score.get("g"),
                    "h": score.get("h"),
                    "f": score.get("f"),
                    "state": "open",
                },
            )
        )

    start_goal_features = [
        node_feature(result["start"], coords, {"role": "start", "id": result["start"]}),
        node_feature(result["goal"], coords, {"role": "goal", "id": result["goal"]}),
    ]

    return {
        "route": {"type": "FeatureCollection", "features": route_features},
        "relaxations": {
            "type": "FeatureCollection",
            "features": relaxation_features,
        },
        "closed": {"type": "FeatureCollection", "features": closed_features},
        "open": {"type": "FeatureCollection", "features": open_features},
        "start_goal": {
            "type": "FeatureCollection",
            "features": start_goal_features,
        },
    }


def make_debug_map_html(debug_layers, meta):
    base_valid_edges = json.loads(Path("edges_valid.geojson").read_text(encoding="utf-8"))
    base_blocked_edges = json.loads(Path("edges_blocked.geojson").read_text(encoding="utf-8"))
    base_land = json.loads(Path("land.geojson").read_text(encoding="utf-8"))
    base_water = json.loads(Path("water.geojson").read_text(encoding="utf-8"))

    center = meta["center"]

    html = """<!doctype html>
<html lang=\"en\">
<head>
  <meta charset=\"utf-8\" />
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\" />
  <title>A* Debug Map</title>
  <link rel=\"stylesheet\" href=\"https://unpkg.com/leaflet@1.9.4/dist/leaflet.css\" integrity=\"sha256-p4NxAoJBhIIN+hmNHrzRCf9tD/miZyoHS5obTRR9BMY=\" crossorigin=\"\"/>
  <style>
    html, body, #map { height: 100%; margin: 0; }
    .stats {
      background: rgba(255, 255, 255, 0.95);
      padding: 8px 10px;
      border-radius: 6px;
      box-shadow: 0 1px 6px rgba(0, 0, 0, 0.25);
      font: 12px/1.35 system-ui, sans-serif;
      max-width: 320px;
    }
  </style>
</head>
<body>
  <div id=\"map\"></div>
  <script src=\"https://unpkg.com/leaflet@1.9.4/dist/leaflet.js\" integrity=\"sha256-20nQCchB9co0qIjJZRGuk2/Z9VM+kNiyxNV1lvTlZBo=\" crossorigin=\"\"></script>
  <script>
    const baseValid = __BASE_VALID__;
    const baseBlocked = __BASE_BLOCKED__;
    const baseLand = __BASE_LAND__;
    const baseWater = __BASE_WATER__;

    const route = __ROUTE__;
    const relax = __RELAX__;
    const closedNodes = __CLOSED__;
    const openNodes = __OPEN__;
    const startGoal = __STARTGOAL__;
    const meta = __META__;

    const map = L.map('map', { preferCanvas: true }).setView([__CENTER_LAT__, __CENTER_LON__], 17);
    L.tileLayer('https://tile.openstreetmap.org/{z}/{x}/{y}.png', {
      maxZoom: 22,
      attribution: '&copy; OpenStreetMap contributors'
    }).addTo(map);

    const waterLayer = L.geoJSON(baseWater, {
      style: { color: '#2a7ab9', weight: 1, fillColor: '#64b5f6', fillOpacity: 0.2 }
    }).addTo(map);

    const landLayer = L.geoJSON(baseLand, {
      style: { color: '#8d6e63', weight: 1, fillColor: '#c8ad7f', fillOpacity: 0.35 }
    }).addTo(map);

    const validEdgesLayer = L.geoJSON(baseValid, {
      style: { color: '#7f8c8d', weight: 0.7, opacity: 0.35 }
    }).addTo(map);

    const blockedEdgesLayer = L.geoJSON(baseBlocked, {
      style: { color: '#d73a49', weight: 1.0, opacity: 0.3, dashArray: '3,4' }
    });

    const relaxLayer = L.geoJSON(relax, {
      style: { color: '#1e88e5', weight: 1.5, opacity: 0.6 },
      onEachFeature: (feature, layer) => {
        const p = feature.properties;
        layer.bindTooltip(
          `relax #${p.idx}<br/>${p.from} -> ${p.to}<br/>g=${p.g.toFixed(6)} h=${p.h.toFixed(6)} f=${p.f.toFixed(6)}`,
          { sticky: true }
        );
      }
    }).addTo(map);

    const routeLayer = L.geoJSON(route, {
      style: { color: '#11a36a', weight: 4, opacity: 0.95 }
    }).addTo(map);

    const closedLayer = L.geoJSON(closedNodes, {
      pointToLayer: (feature, latlng) => L.circleMarker(latlng, {
        radius: 4,
        color: '#000',
        weight: 1,
        fillColor: '#f5f5f5',
        fillOpacity: 0.95
      }),
      onEachFeature: (feature, layer) => {
        const p = feature.properties;
        layer.bindTooltip(
          `closed #${p.order}<br/>node=${p.id}<br/>g=${Number(p.g).toFixed(6)}<br/>h=${Number(p.h).toFixed(6)}<br/>f=${Number(p.f).toFixed(6)}`,
          { sticky: true }
        );
      }
    }).addTo(map);

    const openLayer = L.geoJSON(openNodes, {
      pointToLayer: (feature, latlng) => L.circleMarker(latlng, {
        radius: 4,
        color: '#8c510a',
        weight: 1,
        fillColor: '#f39c12',
        fillOpacity: 0.95
      }),
      onEachFeature: (feature, layer) => {
        const p = feature.properties;
        layer.bindTooltip(
          `open node=${p.id}<br/>g=${Number(p.g).toFixed(6)}<br/>h=${Number(p.h).toFixed(6)}<br/>f=${Number(p.f).toFixed(6)}`,
          { sticky: true }
        );
      }
    }).addTo(map);

    const startGoalLayer = L.geoJSON(startGoal, {
      pointToLayer: (feature, latlng) => {
        const role = feature.properties.role;
        return L.circleMarker(latlng, {
          radius: 7,
          color: role === 'start' ? '#1565c0' : '#b71c1c',
          weight: 2,
          fillColor: role === 'start' ? '#42a5f5' : '#ef5350',
          fillOpacity: 1
        });
      },
      onEachFeature: (feature, layer) => {
        const p = feature.properties;
        layer.bindTooltip(`${p.role}: node ${p.id}`, { sticky: true });
      }
    }).addTo(map);

    L.control.layers(null, {
      'Water': waterLayer,
      'Land blockers': landLayer,
      'All valid edges': validEdgesLayer,
      'Blocked edges': blockedEdgesLayer,
      'A* relaxations': relaxLayer,
      'A* route': routeLayer,
      'Closed set': closedLayer,
      'Open set': openLayer,
      'Start/Goal': startGoalLayer
    }, { collapsed: false }).addTo(map);

    const stats = L.control({ position: 'topright' });
    stats.onAdd = function() {
      const div = L.DomUtil.create('div', 'stats');
      div.innerHTML = [
        '<strong>A* Debug</strong>',
        'heuristic: ' + meta.heuristic,
        'start: ' + meta.start + ' goal: ' + meta.goal,
        'path nodes: ' + meta.path_nodes,
        'path length(deg): ' + (meta.path_length === null ? 'N/A' : meta.path_length.toFixed(6)),
        'expanded nodes: ' + meta.expanded_nodes,
        'relaxations: ' + meta.relaxations,
        'elapsed(s): ' + meta.elapsed_sec.toFixed(4)
      ].join('<br/>');
      return div;
    };
    stats.addTo(map);

    const fitGroup = L.featureGroup([routeLayer, startGoalLayer, closedLayer]);
    if (fitGroup.getBounds().isValid()) {
      map.fitBounds(fitGroup.getBounds().pad(0.1));
    }
  </script>
</body>
</html>
"""

    html = (
        html.replace("__BASE_VALID__", json.dumps(base_valid_edges))
        .replace("__BASE_BLOCKED__", json.dumps(base_blocked_edges))
        .replace("__BASE_LAND__", json.dumps(base_land))
        .replace("__BASE_WATER__", json.dumps(base_water))
        .replace("__ROUTE__", json.dumps(debug_layers["route"]))
        .replace("__RELAX__", json.dumps(debug_layers["relaxations"]))
        .replace("__CLOSED__", json.dumps(debug_layers["closed"]))
        .replace("__OPEN__", json.dumps(debug_layers["open"]))
        .replace("__STARTGOAL__", json.dumps(debug_layers["start_goal"]))
        .replace("__META__", json.dumps(meta))
        .replace("__CENTER_LAT__", f"{center[1]:.10f}")
        .replace("__CENTER_LON__", f"{center[0]:.10f}")
    )

    ASTAR_DEBUG_HTML_PATH.write_text(html, encoding="utf-8")


def run_astar_debug(
    start_id=None,
    goal_id=None,
    heuristic=None,
    heuristic_expr="euclidean",
    max_expansions=None,
):
    g, coords, length_prop = ensure_graph_and_nodes()

    if start_id is None or goal_id is None:
        start_id, goal_id = pick_farthest_nodes(coords)

    result = astar_search(
        g,
        coords,
        start_id=start_id,
        goal_id=goal_id,
        heuristic=heuristic,
        heuristic_expr=heuristic_expr,
        length_prop=length_prop,
        max_expansions=max_expansions,
    )

    debug_layers = build_debug_geojson(result, coords)
    center = (
        sum(lon for lon, _ in coords) / len(coords),
        sum(lat for _, lat in coords) / len(coords),
    )

    meta = {
        "heuristic": result["heuristic_name"],
        "start": result["start"],
        "goal": result["goal"],
        "path_nodes": len(result["path"]),
        "path_length": result["path_length"],
        "expanded_nodes": result["expanded_nodes"],
        "relaxations": len(result["relaxations"]),
        "elapsed_sec": result["elapsed_sec"],
        "center": center,
    }

    payload = {
        "meta": meta,
        "result": result,
        "layers": debug_layers,
    }
    ASTAR_DEBUG_JSON_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")

    make_debug_map_html(debug_layers, meta)

    print("A* done")
    print(f"heuristic: {meta['heuristic']}")
    print(f"start: {meta['start']}  goal: {meta['goal']}")
    print(f"path nodes: {meta['path_nodes']}")
    print(f"path length (deg): {meta['path_length']}")
    print(f"expanded nodes: {meta['expanded_nodes']}")
    print(f"relaxations: {meta['relaxations']}")
    print(f"elapsed sec: {meta['elapsed_sec']:.6f}")
    print(f"saved: {ASTAR_DEBUG_JSON_PATH}")
    print(f"saved: {ASTAR_DEBUG_HTML_PATH}")

    display(IFrame(src=str(ASTAR_DEBUG_HTML_PATH), width="100%", height=640))
    return payload


# --- Example runs ---
# 1) No hardcode needed: choose heuristic expression on the fly.
#    Available variables in expressions: x, y, gx, gy, dx, dy, node, goal
#    Example: "sqrt(dx*dx + dy*dy) * 1.0"
payload_expr = run_astar_debug(heuristic_expr="sqrt(dx*dx + dy*dy)")

# 2) Or pass a callable heuristic dynamically.
# payload_callable = run_astar_debug(heuristic=lambda n, g: 0.8 * euclidean_heuristic(n, g, ensure_graph_and_nodes()[1]))

A* done
heuristic: expr:sqrt(dx*dx + dy*dy)
start: 40  goal: 62
path nodes: 2
path length (deg): 0.003164903461412128
expanded nodes: 2
relaxations: 57
elapsed sec: 0.000766
saved: astar_debug_data.json
saved: astar_debug_map.html


In [ ]:
import json
from pathlib import Path

from IPython.display import IFrame, display


nodes_geo = json.loads(Path("graph_nodes.geojson").read_text(encoding="utf-8"))
valid_geo = json.loads(Path("graph_edges_valid.geojson").read_text(encoding="utf-8"))
blocked_geo = json.loads(Path("graph_edges_blocked.geojson").read_text(encoding="utf-8"))
land_geo = json.loads(Path("graph_land.geojson").read_text(encoding="utf-8"))
water_geo = json.loads(Path("graph_water.geojson").read_text(encoding="utf-8"))

node_ids = [int(feat["properties"]["id"]) for feat in nodes_geo["features"]]
default_start = min(node_ids) if node_ids else 0
default_goal = max(node_ids) if node_ids else 0
default_expr = "sqrt(dx*dx + dy*dy)"

prev_path = Path("astar_debug_data.json")
if prev_path.exists():
    try:
        prev = json.loads(prev_path.read_text(encoding="utf-8"))
        prev_meta = prev.get("meta", {})
        if isinstance(prev_meta.get("start"), int):
            default_start = prev_meta["start"]
        if isinstance(prev_meta.get("goal"), int):
            default_goal = prev_meta["goal"]

        prev_h = prev_meta.get("heuristic", "")
        if isinstance(prev_h, str) and prev_h.startswith("expr:"):
            default_expr = prev_h[len("expr:") :]
        elif isinstance(prev_h, str) and prev_h:
            default_expr = prev_h
    except Exception:
        pass

if default_start not in node_ids and node_ids:
    default_start = node_ids[0]
if default_goal not in node_ids and node_ids:
    default_goal = node_ids[-1]

center_lon = 0.0
center_lat = 0.0
if nodes_geo["features"]:
    center_lon = sum(f["geometry"]["coordinates"][0] for f in nodes_geo["features"]) / len(nodes_geo["features"])
    center_lat = sum(f["geometry"]["coordinates"][1] for f in nodes_geo["features"]) / len(nodes_geo["features"])

html_template = """<!doctype html>
<html lang=\"en\">
<head>
  <meta charset=\"utf-8\" />
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\" />
  <title>A* Interactive Debug Map</title>
  <link rel=\"stylesheet\" href=\"https://unpkg.com/leaflet@1.9.4/dist/leaflet.css\" integrity=\"sha256-p4NxAoJBhIIN+hmNHrzRCf9tD/miZyoHS5obTRR9BMY=\" crossorigin=\"\"/>
  <style>
    html, body, #map { height: 100%; margin: 0; }
    .panel {
      position: absolute;
      top: 12px;
      left: 12px;
      z-index: 1200;
      width: min(360px, calc(100vw - 24px));
      max-height: calc(100vh - 24px);
      overflow: auto;
      background: rgba(255, 255, 255, 0.96);
      border: 1px solid rgba(0, 0, 0, 0.14);
      border-radius: 10px;
      box-shadow: 0 4px 20px rgba(0, 0, 0, 0.24);
      padding: 10px;
      font: 12px/1.4 system-ui, sans-serif;
    }
    .panel h3 {
      margin: 0 0 8px;
      font-size: 14px;
    }
    .grid {
      display: grid;
      grid-template-columns: 1fr 1fr;
      gap: 6px;
      align-items: center;
      margin-bottom: 8px;
    }
    .grid label {
      font-weight: 600;
    }
    .grid input, .grid select {
      width: 100%;
      box-sizing: border-box;
      padding: 4px 6px;
      border: 1px solid #bbb;
      border-radius: 5px;
      font: inherit;
    }
    .heur-row {
      display: grid;
      grid-template-columns: 1fr;
      gap: 6px;
      margin-bottom: 8px;
    }
    .heur-row input {
      width: 100%;
      box-sizing: border-box;
      padding: 5px 6px;
      border: 1px solid #bbb;
      border-radius: 5px;
      font: inherit;
    }
    .btn-row {
      display: flex;
      flex-wrap: wrap;
      gap: 6px;
      margin-bottom: 8px;
    }
    .btn-row button {
      border: 1px solid #777;
      border-radius: 6px;
      background: #f7f7f7;
      padding: 5px 8px;
      cursor: pointer;
      font: inherit;
    }
    .btn-row button.primary {
      background: #1666d3;
      border-color: #1356af;
      color: white;
      font-weight: 600;
    }
    .muted {
      color: #444;
      font-size: 11px;
    }
    .message {
      min-height: 16px;
      font-size: 11px;
      margin: 4px 0 8px;
    }
    .message.ok { color: #0b7c3b; }
    .message.err { color: #b00020; }
    .stats {
      border-top: 1px solid #ddd;
      padding-top: 8px;
      font-size: 11px;
      white-space: pre-line;
    }
  </style>
</head>
<body>
  <div id=\"map\"></div>

  <div class=\"panel\">
    <h3>A* Debug Controls</h3>
    <div class=\"grid\">
      <label for=\"startNode\">Start node</label>
      <input id=\"startNode\" type=\"number\" step=\"1\" value=\"__DEFAULT_START__\" />
      <label for=\"goalNode\">Goal node</label>
      <input id=\"goalNode\" type=\"number\" step=\"1\" value=\"__DEFAULT_GOAL__\" />
      <label for=\"maxExp\">Max expansions</label>
      <input id=\"maxExp\" type=\"number\" step=\"1\" placeholder=\"optional\" />
    </div>

    <div class=\"heur-row\">
      <label for=\"heurExpr\"><strong>h()</strong> expression</label>
      <input id=\"heurExpr\" value=\"__DEFAULT_EXPR__\" />
      <div class=\"muted\">Variables: x, y, gx, gy, dx, dy, node, goal. Presets: euclidean, manhattan, zero.</div>
    </div>

    <div class=\"btn-row\">
      <button id=\"runBtn\" class=\"primary\">Run A*</button>
      <button id=\"swapBtn\">Swap</button>
      <button id=\"pickStartBtn\">Pick start on map</button>
      <button id=\"pickGoalBtn\">Pick goal on map</button>
      <button id=\"resetPickBtn\">Cancel pick</button>
    </div>

    <div id=\"msg\" class=\"message\"></div>
    <div id=\"stats\" class=\"stats\">No run yet.</div>
  </div>

  <script src=\"https://unpkg.com/leaflet@1.9.4/dist/leaflet.js\" integrity=\"sha256-20nQCchB9co0qIjJZRGuk2/Z9VM+kNiyxNV1lvTlZBo=\" crossorigin=\"\"></script>
  <script>
    const nodesGeo = __NODES__;
    const validGeo = __VALID__;
    const blockedGeo = __BLOCKED__;
    const landGeo = __LAND__;
    const waterGeo = __WATER__;

    const map = L.map('map', { preferCanvas: true }).setView([__CENTER_LAT__, __CENTER_LON__], 17);
    L.tileLayer('https://tile.openstreetmap.org/{z}/{x}/{y}.png', {
      maxZoom: 22,
      attribution: '&copy; OpenStreetMap contributors'
    }).addTo(map);

    const waterLayer = L.geoJSON(waterGeo, {
      style: { color: '#2a7ab9', weight: 1, fillColor: '#64b5f6', fillOpacity: 0.22 }
    }).addTo(map);

    const landLayer = L.geoJSON(landGeo, {
      style: { color: '#8d6e63', weight: 1, fillColor: '#c8ad7f', fillOpacity: 0.36 }
    }).addTo(map);

    const validEdgesLayer = L.geoJSON(validGeo, {
      style: { color: '#7f8c8d', weight: 0.8, opacity: 0.4 }
    }).addTo(map);

    const blockedEdgesLayer = L.geoJSON(blockedGeo, {
      style: { color: '#d73a49', weight: 1, opacity: 0.3, dashArray: '3,4' }
    });

    const routeLayer = L.geoJSON(null, {
      style: { color: '#11a36a', weight: 4, opacity: 0.95 }
    }).addTo(map);

    const relaxLayer = L.geoJSON(null, {
      style: { color: '#1e88e5', weight: 1.5, opacity: 0.65 },
      onEachFeature: (feature, layer) => {
        const p = feature.properties;
        layer.bindTooltip(`relax #${p.idx}<br/>${p.from} -> ${p.to}<br/>g=${p.g.toFixed(6)} h=${p.h.toFixed(6)} f=${p.f.toFixed(6)}`, { sticky: true });
      }
    }).addTo(map);

    const closedLayer = L.geoJSON(null, {
      pointToLayer: (feature, latlng) => L.circleMarker(latlng, {
        radius: 4,
        color: '#000',
        weight: 1,
        fillColor: '#f5f5f5',
        fillOpacity: 0.95
      }),
      onEachFeature: (feature, layer) => {
        const p = feature.properties;
        layer.bindTooltip(`closed #${p.order}<br/>node=${p.id}<br/>g=${Number(p.g).toFixed(6)}<br/>h=${Number(p.h).toFixed(6)}<br/>f=${Number(p.f).toFixed(6)}`, { sticky: true });
      }
    }).addTo(map);

    const openLayer = L.geoJSON(null, {
      pointToLayer: (feature, latlng) => L.circleMarker(latlng, {
        radius: 4,
        color: '#8c510a',
        weight: 1,
        fillColor: '#f39c12',
        fillOpacity: 0.95
      }),
      onEachFeature: (feature, layer) => {
        const p = feature.properties;
        layer.bindTooltip(`open node=${p.id}<br/>g=${Number(p.g).toFixed(6)}<br/>h=${Number(p.h).toFixed(6)}<br/>f=${Number(p.f).toFixed(6)}`, { sticky: true });
      }
    }).addTo(map);

    const startGoalLayer = L.geoJSON(null, {
      pointToLayer: (feature, latlng) => {
        const role = feature.properties.role;
        return L.circleMarker(latlng, {
          radius: 7,
          color: role === 'start' ? '#1565c0' : '#b71c1c',
          weight: 2,
          fillColor: role === 'start' ? '#42a5f5' : '#ef5350',
          fillOpacity: 1
        });
      },
      onEachFeature: (feature, layer) => {
        const p = feature.properties;
        layer.bindTooltip(`${p.role}: node ${p.id}`, { sticky: true });
      }
    }).addTo(map);

    let pickMode = null;

    const nodes = nodesGeo.features.map((f) => ({
      id: Number(f.properties.id),
      lon: Number(f.geometry.coordinates[0]),
      lat: Number(f.geometry.coordinates[1])
    }));

    const nodeById = new Map(nodes.map((n) => [n.id, n]));
    const adjacency = new Map(nodes.map((n) => [n.id, []]));

    for (const feat of validGeo.features) {
      const p = feat.properties || {};
      const src = Number(p.source);
      const dst = Number(p.target);
      if (!adjacency.has(src) || !adjacency.has(dst)) continue;

      const coords = feat.geometry.coordinates;
      let w = Number(p.length_deg);
      if (!Number.isFinite(w)) {
        const dx = Number(coords[1][0]) - Number(coords[0][0]);
        const dy = Number(coords[1][1]) - Number(coords[0][1]);
        w = Math.hypot(dx, dy);
      }

      adjacency.get(src).push({ to: dst, w });
      adjacency.get(dst).push({ to: src, w });
    }

    const allNodesLayer = L.geoJSON(nodesGeo, {
      pointToLayer: (feature, latlng) => L.circleMarker(latlng, {
        radius: 3,
        color: '#1f1f1f',
        weight: 1,
        fillColor: '#fef08a',
        fillOpacity: 0.8
      }),
      onEachFeature: (feature, layer) => {
        const id = Number(feature.properties.id);
        layer.bindTooltip(`node ${id}`, { sticky: true });
        layer.on('click', () => {
          if (pickMode === 'start') {
            startInput.value = String(id);
            setMessage(`Picked start node ${id}`, false);
            pickMode = null;
          } else if (pickMode === 'goal') {
            goalInput.value = String(id);
            setMessage(`Picked goal node ${id}`, false);
            pickMode = null;
          }
        });
      }
    }).addTo(map);

    L.control.layers(null, {
      'Water': waterLayer,
      'Land blockers': landLayer,
      'All valid edges': validEdgesLayer,
      'Blocked edges': blockedEdgesLayer,
      'All nodes': allNodesLayer,
      'A* relaxations': relaxLayer,
      'A* route': routeLayer,
      'Closed set': closedLayer,
      'Open set': openLayer,
      'Start/Goal': startGoalLayer
    }, { collapsed: false }).addTo(map);

    class MinHeap {
      constructor() {
        this.data = [];
      }
      get size() {
        return this.data.length;
      }
      push(item) {
        this.data.push(item);
        this._up(this.data.length - 1);
      }
      pop() {
        if (!this.data.length) return null;
        const root = this.data[0];
        const end = this.data.pop();
        if (this.data.length) {
          this.data[0] = end;
          this._down(0);
        }
        return root;
      }
      _less(a, b) {
        return a.f < b.f || (a.f === b.f && a.seq < b.seq);
      }
      _up(idx) {
        while (idx > 0) {
          const p = Math.floor((idx - 1) / 2);
          if (!this._less(this.data[idx], this.data[p])) break;
          [this.data[idx], this.data[p]] = [this.data[p], this.data[idx]];
          idx = p;
        }
      }
      _down(idx) {
        const n = this.data.length;
        while (true) {
          let left = idx * 2 + 1;
          let right = left + 1;
          let smallest = idx;
          if (left < n && this._less(this.data[left], this.data[smallest])) smallest = left;
          if (right < n && this._less(this.data[right], this.data[smallest])) smallest = right;
          if (smallest === idx) break;
          [this.data[idx], this.data[smallest]] = [this.data[smallest], this.data[idx]];
          idx = smallest;
        }
      }
    }

    function compileHeuristic(exprRaw) {
      const text = String(exprRaw || '').trim();
      const lowered = text.toLowerCase();

      if (!text || lowered === 'euclidean' || lowered === 'l2') {
        return {
          label: 'euclidean',
          fn: (nodeId, goalId) => {
            const n = nodeById.get(nodeId);
            const g = nodeById.get(goalId);
            return Math.hypot(g.lon - n.lon, g.lat - n.lat);
          }
        };
      }

      if (lowered === 'manhattan' || lowered === 'l1') {
        return {
          label: 'manhattan',
          fn: (nodeId, goalId) => {
            const n = nodeById.get(nodeId);
            const g = nodeById.get(goalId);
            return Math.abs(g.lon - n.lon) + Math.abs(g.lat - n.lat);
          }
        };
      }

      if (lowered === 'zero' || lowered === 'dijkstra') {
        return {
          label: 'zero',
          fn: () => 0
        };
      }

      const allowedIdentifiers = new Set([
        'x', 'y', 'gx', 'gy', 'dx', 'dy', 'node', 'goal',
        'Math',
        'sqrt', 'abs', 'pow', 'sin', 'cos', 'tan', 'asin', 'acos',
        'atan', 'atan2', 'exp', 'log', 'log10', 'min', 'max', 'hypot',
        'pi', 'e'
      ]);

      if (text.includes(';') || text.includes('{') || text.includes('}') || text.includes('[') || text.includes(']')) {
        throw new Error('h() must be a pure expression, not statements or blocks.');
      }

      const disallowedPattern = /(=>|function\\b|new\\s+|this\\b|window\\b|globalThis\\b|document\\b|constructor\\b|__proto__|prototype\\b)/;
      if (disallowedPattern.test(text)) {
        throw new Error('Unsupported token in h() expression.');
      }

      const idMatches = text.match(/[A-Za-z_]\\w*/g) || [];
      const badIds = [...new Set(idMatches.filter((name) => !allowedIdentifiers.has(name)))];
      if (badIds.length) {
        throw new Error(`Unsupported names in h(): ${badIds.join(', ')}`);
      }

      let exprFn;
      try {
        exprFn = new Function(
          'x',
          'y',
          'gx',
          'gy',
          'dx',
          'dy',
          'node',
          'goal',
          'Math',
          'sqrt',
          'abs',
          'pow',
          'sin',
          'cos',
          'tan',
          'asin',
          'acos',
          'atan',
          'atan2',
          'exp',
          'log',
          'log10',
          'min',
          'max',
          'hypot',
          'pi',
          'e',
          `"use strict"; return (${text});`
        );
      } catch (err) {
        throw new Error(`Invalid h() expression: ${err.message}`);
      }

      return {
        label: `expr:${text}`,
        fn: (nodeId, goalId) => {
          const n = nodeById.get(nodeId);
          const g = nodeById.get(goalId);
          const dx = Math.abs(g.lon - n.lon);
          const dy = Math.abs(g.lat - n.lat);
          let value;
          try {
            value = Number(
              exprFn(
                n.lon,
                n.lat,
                g.lon,
                g.lat,
                dx,
                dy,
                nodeId,
                goalId,
                Math,
                Math.sqrt,
                Math.abs,
                Math.pow,
                Math.sin,
                Math.cos,
                Math.tan,
                Math.asin,
                Math.acos,
                Math.atan,
                Math.atan2,
                Math.exp,
                Math.log,
                Math.log10,
                Math.min,
                Math.max,
                Math.hypot,
                Math.PI,
                Math.E
              )
            );
          } catch (err) {
            throw new Error(`Error while evaluating h() at node ${nodeId}: ${err.message}`);
          }
          if (!Number.isFinite(value)) {
            throw new Error(`h() produced non-finite value at node ${nodeId}`);
          }
          return value;
        }
      };
    }

    function toPointFeature(nodeId, props) {
      const n = nodeById.get(nodeId);
      return {
        type: 'Feature',
        geometry: { type: 'Point', coordinates: [n.lon, n.lat] },
        properties: props || {}
      };
    }

    function toEdgeFeature(src, dst, props) {
      const s = nodeById.get(src);
      const t = nodeById.get(dst);
      return {
        type: 'Feature',
        geometry: {
          type: 'LineString',
          coordinates: [[s.lon, s.lat], [t.lon, t.lat]]
        },
        properties: props || {}
      };
    }

    function runAStar(startId, goalId, heuristicExpr, maxExpansions) {
      if (!nodeById.has(startId)) throw new Error(`Start node ${startId} does not exist`);
      if (!nodeById.has(goalId)) throw new Error(`Goal node ${goalId} does not exist`);

      const { label, fn: h } = compileHeuristic(heuristicExpr);
      const openHeap = new MinHeap();

      const gScore = new Map([[startId, 0]]);
      const hScore = new Map([[startId, h(startId, goalId)]]);
      const fScore = new Map([[startId, gScore.get(startId) + hScore.get(startId)]]);
      const parent = new Map([[startId, null]]);

      const openSet = new Set([startId]);
      const closedSet = new Set();
      const closedOrder = [];
      const relaxations = [];

      let seq = 0;
      openHeap.push({ f: fScore.get(startId), seq, id: startId });

      const t0 = performance.now();
      let expansions = 0;

      while (openHeap.size > 0) {
        const item = openHeap.pop();
        const current = item.id;

        if (closedSet.has(current)) continue;

        openSet.delete(current);
        closedSet.add(current);
        closedOrder.push(current);

        if (current === goalId) break;

        expansions += 1;
        if (Number.isFinite(maxExpansions) && maxExpansions >= 0 && expansions > maxExpansions) {
          break;
        }

        const neighbors = adjacency.get(current) || [];
        for (const nb of neighbors) {
          const next = nb.to;
          if (closedSet.has(next)) continue;

          const tentativeG = gScore.get(current) + nb.w;
          const bestKnown = gScore.has(next) ? gScore.get(next) : Infinity;
          if (tentativeG >= bestKnown) continue;

          parent.set(next, current);
          gScore.set(next, tentativeG);

          const hVal = h(next, goalId);
          hScore.set(next, hVal);

          const fVal = tentativeG + hVal;
          fScore.set(next, fVal);

          seq += 1;
          openHeap.push({ f: fVal, seq, id: next });
          openSet.add(next);

          relaxations.push({
            from: current,
            to: next,
            g: tentativeG,
            h: hVal,
            f: fVal,
            step_cost: nb.w
          });
        }
      }

      const elapsedSec = (performance.now() - t0) / 1000;

      const path = [];
      if (startId === goalId) {
        path.push(startId);
      } else if (parent.has(goalId)) {
        let cur = goalId;
        while (cur !== null) {
          path.push(cur);
          cur = parent.get(cur);
        }
        path.reverse();
      }

      const pathEdges = [];
      for (let i = 0; i + 1 < path.length; i += 1) {
        pathEdges.push([path[i], path[i + 1]]);
      }

      const scoreKeys = new Set([...gScore.keys(), ...openSet, ...closedSet]);
      const scores = {};
      for (const id of scoreKeys) {
        scores[id] = {
          g: gScore.has(id) ? gScore.get(id) : Infinity,
          h: hScore.has(id) ? hScore.get(id) : Infinity,
          f: fScore.has(id) ? fScore.get(id) : Infinity,
          parent: parent.has(id) ? parent.get(id) : null,
          closed: closedSet.has(id),
          open: openSet.has(id)
        };
      }

      return {
        heuristic_name: label,
        start: startId,
        goal: goalId,
        elapsed_sec: elapsedSec,
        expanded_nodes: closedOrder.length,
        closed_order: closedOrder,
        open_remaining: [...openSet].sort((a, b) => a - b),
        relaxations,
        path,
        path_edges: pathEdges,
        path_length: gScore.has(goalId) ? gScore.get(goalId) : null,
        scores
      };
    }

    function renderResult(result) {
      const routeFeatures = result.path_edges.map((e, idx) =>
        toEdgeFeature(e[0], e[1], { step: idx, source: e[0], target: e[1] })
      );

      const relaxFeatures = result.relaxations.map((r, idx) =>
        toEdgeFeature(r.from, r.to, {
          idx,
          from: r.from,
          to: r.to,
          g: r.g,
          h: r.h,
          f: r.f,
          step_cost: r.step_cost
        })
      );

      const closedFeatures = result.closed_order.map((id, order) => {
        const s = result.scores[id] || {};
        return toPointFeature(id, {
          id,
          order,
          g: s.g,
          h: s.h,
          f: s.f,
          state: 'closed'
        });
      });

      const openFeatures = result.open_remaining.map((id) => {
        const s = result.scores[id] || {};
        return toPointFeature(id, {
          id,
          g: s.g,
          h: s.h,
          f: s.f,
          state: 'open'
        });
      });

      const sgFeatures = [
        toPointFeature(result.start, { role: 'start', id: result.start }),
        toPointFeature(result.goal, { role: 'goal', id: result.goal })
      ];

      routeLayer.clearLayers();
      relaxLayer.clearLayers();
      closedLayer.clearLayers();
      openLayer.clearLayers();
      startGoalLayer.clearLayers();

      routeLayer.addData({ type: 'FeatureCollection', features: routeFeatures });
      relaxLayer.addData({ type: 'FeatureCollection', features: relaxFeatures });
      closedLayer.addData({ type: 'FeatureCollection', features: closedFeatures });
      openLayer.addData({ type: 'FeatureCollection', features: openFeatures });
      startGoalLayer.addData({ type: 'FeatureCollection', features: sgFeatures });

      const group = L.featureGroup([routeLayer, startGoalLayer, closedLayer]);
      if (group.getBounds().isValid()) {
        map.fitBounds(group.getBounds().pad(0.12));
      }

      const stats = [
        `heuristic: ${result.heuristic_name}`,
        `start: ${result.start} goal: ${result.goal}`,
        `path nodes: ${result.path.length}`,
        `path length (deg): ${result.path_length === null ? 'N/A' : result.path_length.toFixed(6)}`,
        `expanded nodes: ${result.expanded_nodes}`,
        `relaxations: ${result.relaxations.length}`,
        `elapsed (s): ${result.elapsed_sec.toFixed(6)}`
      ];
      statsEl.textContent = stats.join('\n');
    }

    function nearestNodeId(lon, lat) {
      let bestId = null;
      let bestD = Infinity;
      for (const n of nodes) {
        const dx = n.lon - lon;
        const dy = n.lat - lat;
        const d = dx * dx + dy * dy;
        if (d < bestD) {
          bestD = d;
          bestId = n.id;
        }
      }
      return bestId;
    }

    function setMessage(text, isError) {
      msgEl.textContent = text;
      msgEl.className = isError ? 'message err' : 'message ok';
    }

    const startInput = document.getElementById('startNode');
    const goalInput = document.getElementById('goalNode');
    const maxExpInput = document.getElementById('maxExp');
    const heurExprInput = document.getElementById('heurExpr');
    const msgEl = document.getElementById('msg');
    const statsEl = document.getElementById('stats');

    document.getElementById('runBtn').addEventListener('click', () => {
      const startId = Number(startInput.value);
      const goalId = Number(goalInput.value);
      const heurExpr = heurExprInput.value;
      const maxExpRaw = maxExpInput.value.trim();
      const maxExp = maxExpRaw === '' ? null : Number(maxExpRaw);

      try {
        const result = runAStar(startId, goalId, heurExpr, maxExp);
        renderResult(result);
        setMessage('A* run completed.', false);
      } catch (err) {
        setMessage(err.message, true);
      }
    });

    document.getElementById('swapBtn').addEventListener('click', () => {
      const tmp = startInput.value;
      startInput.value = goalInput.value;
      goalInput.value = tmp;
      setMessage('Swapped start and goal.', false);
    });

    document.getElementById('pickStartBtn').addEventListener('click', () => {
      pickMode = 'start';
      setMessage('Pick mode: click a node marker or map to set START.', false);
    });

    document.getElementById('pickGoalBtn').addEventListener('click', () => {
      pickMode = 'goal';
      setMessage('Pick mode: click a node marker or map to set GOAL.', false);
    });

    document.getElementById('resetPickBtn').addEventListener('click', () => {
      pickMode = null;
      setMessage('Pick mode canceled.', false);
    });

    map.on('click', (e) => {
      if (!pickMode) return;
      const nearest = nearestNodeId(e.latlng.lng, e.latlng.lat);
      if (nearest === null) return;

      if (pickMode === 'start') {
        startInput.value = String(nearest);
        setMessage(`Picked start node ${nearest}`, false);
      } else {
        goalInput.value = String(nearest);
        setMessage(`Picked goal node ${nearest}`, false);
      }
      pickMode = null;
    });

    // Initial run on load.
    try {
      const initResult = runAStar(Number(startInput.value), Number(goalInput.value), heurExprInput.value, null);
      renderResult(initResult);
      setMessage('Initial A* run completed.', false);
    } catch (err) {
      setMessage(err.message, true);
    }
  </script>
</body>
</html>
"""

html = (
    html_template.replace("__NODES__", json.dumps(nodes_geo))
    .replace("__VALID__", json.dumps(valid_geo))
    .replace("__BLOCKED__", json.dumps(blocked_geo))
    .replace("__LAND__", json.dumps(land_geo))
    .replace("__WATER__", json.dumps(water_geo))
    .replace("__CENTER_LAT__", f"{center_lat:.10f}")
    .replace("__CENTER_LON__", f"{center_lon:.10f}")
    .replace("__DEFAULT_START__", str(default_start))
    .replace("__DEFAULT_GOAL__", str(default_goal))
    .replace("__DEFAULT_EXPR__", default_expr)
)

Path("astar_debug_map.html").write_text(html, encoding="utf-8")
print("Wrote interactive debug map with heuristic/start/goal controls: astar_debug_map.html")
display(IFrame(src="astar_debug_map.html", width="100%", height=680))

Wrote interactive debug map with heuristic/start/goal controls: astar_debug_map.html


In [ ]:
from pathlib import Path
from IPython.display import IFrame, display

html_path = Path("astar_debug_map.html")
if not html_path.exists():
    raise FileNotFoundError("astar_debug_map.html was not found. Run the A* debug generation cell first.")

print("Interactive debug map already supports dynamic h() expressions directly (no post-patch needed).")
display(IFrame(src="astar_debug_map.html", width="100%", height=680))

Interactive debug map already supports dynamic h() expressions directly (no post-patch needed).


website for geojson check: https://dropchop.io/